# Datasource Instance Deep Scanner

Finds all Fabric items using specific **datasource IDs** and retrieves full connection details  
(server names, connection strings, URLs) including whether the datasource is used as **source or sink**.

---

## Required Access Privileges

### For a User Identity (interactive notebook execution):

| Requirement | Details |
|-------------|---------|
| **Microsoft Entra ID Role** | **Fabric Administrator** (or Power BI Service Administrator) |
| **Why** | Required for Admin Scanner API (`/admin/workspaces/getInfo`) and Admin Groups API (`/admin/groups`) to scan all tenant workspaces |
| **Fabric Items API** | `getDefinition` requires at least **Contributor** role on each workspace being scanned. Fabric Admins can read definitions across all workspaces. |

### For a Service Principal (automated/scheduled execution):

| Requirement | Details |
|-------------|---------|
| **Tenant Setting #1** | Enable: *"Service principals can use read-only admin APIs"* (Admin Portal → Tenant Settings → Admin API settings) |
| **Tenant Setting #2** | Enable: *"Service principals can access Fabric APIs"* (Admin Portal → Tenant Settings → Developer settings) |
| **Tenant Setting #3** | Enable: *"Enhance admin APIs responses with detailed metadata"* — required for `getDefinition` via SPN |
| **Security Group** | Add the SPN to the security group allowed in the above tenant settings |
| **API Permission** | `Tenant.Read.All` (Application permission) on Power BI Service API in Azure Portal → App Registrations → API Permissions |
| **No admin-consent-required permissions** | The SPN must NOT have any admin-consent-required Power BI permissions set in Azure Portal (per Scanner API docs) |

### Minimum Permissions per API Used:

| API | Endpoint | Required Permission |
|-----|----------|-------------------|
| Admin Groups | `GET /v1.0/myorg/admin/groups` | Fabric Admin role OR SPN with `Tenant.Read.All` + tenant settings |
| Scanner API (trigger) | `POST /v1.0/myorg/admin/workspaces/getInfo` | Fabric Admin role OR SPN with `Tenant.Read.All` + tenant settings |
| Scanner API (poll) | `GET /v1.0/myorg/admin/workspaces/scanStatus/{id}` | Same as above |
| Scanner API (result) | `GET /v1.0/myorg/admin/workspaces/scanResult/{id}` | Same as above |
| Items API | `POST /v1/workspaces/{ws}/items/{id}/getDefinition` | Fabric Admin OR Contributor/Admin on workspace |

### Notes:
- Without **Fabric Administrator** role, the notebook will only scan workspaces where the user has membership
- `getDefinition` may return HTTP 403 for workspaces where the caller lacks Contributor access
- The Scanner API has a rate limit of **500 requests/hour** — the notebook's batch mode handles this
- Token audience: `https://analysis.windows.net/powerbi/api` (PBI) — used by `mssparkutils.credentials.getToken("pbi")`

---

## Approach
1. **Scanner API** (`getInfo` with `datasourceDetails=true`) — scans workspaces in batches to find which ones contain the target datasource IDs
2. **getDefinition API** — for matching items, retrieves artifact definitions to extract source/sink details
3. Writes raw Scanner API output (matching workspaces) to `scanner_workspace_raw`
4. Writes item-level details to `scanner_item_datasource_details`


In [ ]:
# ============================================================
# Configuration
# ============================================================

# TARGET DATASOURCE IDs to search for
TARGET_DATASOURCE_IDS = [
    "92b5963e-05c1-4d87-94cb-96625efec6e5",
    "b1c0fcef-d98e-4cc9-9025-d76bda9ddbc2",
    "acf72318-30ed-4e8b-965e-392babdabdcb"   
    # Add more datasource IDs here
]

# Optional: limit scan to specific workspace IDs (None = scan entire tenant)
WORKSPACE_IDS = None  # e.g., ["ws-guid-1", "ws-guid-2"] or None for all

# Batch settings
SCANNER_BATCH_SIZE = 100  # Max per Scanner API call (API limit)
SCAN_PAUSE_SECONDS = 2  # Pause between scanner batches
DEFINITION_BATCH_SIZE = 10  # Items to scan via getDefinition per batch
DEFINITION_PAUSE_SECONDS = 1  # Pause between definition batches

# Retry settings
MAX_RETRIES = 5
POLL_INTERVAL_SECONDS = 2
POLL_TIMEOUT_SECONDS = 120
REQUEST_TIMEOUT = 60

# Output tables
RAW_WORKSPACE_TABLE = "scanner_workspace_raw"
ITEM_DETAILS_TABLE = "scanner_item_datasource_details"

# API endpoints
FABRIC_BASE = "https://api.fabric.microsoft.com"
PBI_BASE = "https://api.powerbi.com"

# Item types whose definitions can be scanned for source/sink
SCANNABLE_ITEM_TYPES = ["DataPipeline", "Dataflow", "CopyJob"]


StatementMeta(, 03250aa0-f0f9-495f-9be1-46982cff225c, 3, Finished, Available, Finished, False)

In [ ]:
# ============================================================
# Imports + Authentication
# ============================================================
import base64
import json
import math
import re
import time
from datetime import datetime, timezone

import requests
from pyspark.sql import functions as F
from pyspark.sql import types as T

try:
    from notebookutils import mssparkutils
except Exception:
    import notebookutils
    mssparkutils = notebookutils.mssparkutils

TOKEN = mssparkutils.credentials.getToken("pbi")
HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "User-Agent": "datasource-deep-scanner/1.0",
}

def refresh_token():
    global TOKEN, HEADERS
    TOKEN = mssparkutils.credentials.getToken("pbi")
    HEADERS["Authorization"] = f"Bearer {TOKEN}"
    print(f"  [auth] Token refreshed at {datetime.now(timezone.utc).isoformat()}")

def utc_now():
    return datetime.now(timezone.utc)

RUN_TS = utc_now()
print(f"Authenticated at {RUN_TS.isoformat()}")
print(f"Searching for {len(TARGET_DATASOURCE_IDS)} datasource ID(s)")


StatementMeta(, 03250aa0-f0f9-495f-9be1-46982cff225c, 4, Finished, Available, Finished, False)

Authenticated at 2026-05-27T21:12:09.476032+00:00
Searching for 3 datasource ID(s)


In [ ]:
# ============================================================
# HTTP helpers with error handling (400, 401, 429, 5xx)
# ============================================================
import random

def api_request(method, url, body=None, retries=MAX_RETRIES):
    """Make an HTTP request with full retry logic for 400/401/429/5xx."""
    for attempt in range(retries):
        try:
            if method == "GET":
                resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            elif method == "POST":
                resp = requests.post(url, headers=HEADERS, json=body, timeout=REQUEST_TIMEOUT)
            else:
                raise ValueError(f"Unsupported method: {method}")
            
            if resp.status_code == 200:
                return resp.json() if resp.text else {}, None
            if resp.status_code == 202:
                return resp, None  # LRO - return raw response
            
            # 401 - Token expired
            if resp.status_code == 401:
                refresh_token()
                continue
            
            # 429 - Throttled
            if resp.status_code == 429:
                retry_after = float(resp.headers.get("Retry-After", 60))
                print(f"  [throttle] 429 on {url[:80]}... waiting {retry_after}s (attempt {attempt+1}/{retries})")
                time.sleep(retry_after)
                continue
            
            # 400 - Bad request (don't retry)
            if resp.status_code == 400:
                return None, f"HTTP 400: {resp.text[:300]}"
            
            # 403/404 - Don't retry
            if resp.status_code in (403, 404):
                return None, f"HTTP {resp.status_code}: {resp.text[:200]}"
            
            # 5xx - Retry with backoff
            if resp.status_code >= 500:
                wait = min(30, (2 ** attempt) + random.random())
                print(f"  [error] HTTP {resp.status_code} on {url[:80]}... retrying in {wait:.1f}s")
                time.sleep(wait)
                continue
            
            return None, f"HTTP {resp.status_code}: {resp.text[:200]}"
        
        except requests.exceptions.Timeout:
            wait = min(30, (2 ** attempt) + random.random())
            print(f"  [timeout] {url[:80]}... retrying in {wait:.1f}s")
            time.sleep(wait)
        except Exception as exc:
            return None, f"Exception: {str(exc)[:200]}"
    
    return None, f"Max retries ({retries}) exceeded for {url[:100]}"

def poll_lro(location_url):
    """Poll an LRO until complete."""
    deadline = time.time() + POLL_TIMEOUT_SECONDS
    while time.time() < deadline:
        result, err = api_request("GET", location_url)
        if err:
            return None, err
        if isinstance(result, dict):
            status = (result.get("status") or result.get("state") or "").lower()
            if status in ("succeeded", "success", "completed"):
                return result, None
            if status in ("failed", "cancelled"):
                return None, f"LRO {status}: {json.dumps(result)[:300]}"
        time.sleep(POLL_INTERVAL_SECONDS)
    return None, "LRO timeout"

print("HTTP helpers ready")


StatementMeta(, 03250aa0-f0f9-495f-9be1-46982cff225c, 5, Finished, Available, Finished, False)

HTTP helpers ready


In [ ]:
# ============================================================
# Step 1: Get workspace IDs to scan
# ============================================================

if WORKSPACE_IDS:
    workspace_ids = list(WORKSPACE_IDS)
    print(f"Using {len(workspace_ids)} specified workspace IDs")
else:
    workspace_ids = []
    url = f"{PBI_BASE}/v1.0/myorg/admin/groups?%24top=5000"
    while url:
        data, err = api_request("GET", url)
        if err:
            print(f"  ERROR listing workspaces: {err}")
            break
        for ws in data.get("value", []):
            workspace_ids.append(ws["id"])
        url = data.get("@odata.nextLink")
    print(f"Found {len(workspace_ids)} workspaces in tenant")


StatementMeta(, 03250aa0-f0f9-495f-9be1-46982cff225c, 6, Finished, Available, Finished, False)

Found 73 workspaces in tenant


In [ ]:
# ============================================================
# Step 2: Run Scanner API in batches to find target datasource IDs
# ============================================================

target_set = set(d.lower() for d in TARGET_DATASOURCE_IDS)
matching_workspaces_raw = []  # Full raw workspace data for matches
matching_artifacts = []  # Artifacts with datasourceUsages matching target
workspace_datasource_map = {}  # ws_id -> list of matching datasource details

total_batches = math.ceil(len(workspace_ids) / SCANNER_BATCH_SIZE)
ws_scanned = 0
ws_with_matches = 0

print(f"Scanning {len(workspace_ids)} workspaces in {total_batches} batches...")
print(f"Looking for datasource IDs: {TARGET_DATASOURCE_IDS}")

for batch_idx in range(total_batches):
    batch_start = batch_idx * SCANNER_BATCH_SIZE
    batch_end = min(batch_start + SCANNER_BATCH_SIZE, len(workspace_ids))
    batch_ids = workspace_ids[batch_start:batch_end]
    
    # Trigger scan
    trigger_url = f"{PBI_BASE}/v1.0/myorg/admin/workspaces/getInfo?datasourceDetails=true"
    trigger_result, err = api_request("POST", trigger_url, body={"workspaces": batch_ids})
    
    if err:
        print(f"  Batch {batch_idx+1}/{total_batches}: Trigger failed - {err}")
        time.sleep(SCAN_PAUSE_SECONDS)
        continue
    
    # Get scan ID
    if isinstance(trigger_result, dict):
        scan_id = trigger_result.get("id")
    else:
        # 202 response object
        scan_id = trigger_result.json().get("id") if hasattr(trigger_result, 'json') else None
    
    if not scan_id:
        print(f"  Batch {batch_idx+1}: No scan ID returned")
        continue
    
    # Poll for completion
    status_url = f"{PBI_BASE}/v1.0/myorg/admin/workspaces/scanStatus/{scan_id}"
    poll_result, poll_err = None, None
    deadline = time.time() + POLL_TIMEOUT_SECONDS
    while time.time() < deadline:
        status_data, status_err = api_request("GET", status_url)
        if status_err:
            poll_err = status_err
            break
        status = (status_data or {}).get("status", "")
        if status == "Succeeded":
            poll_result = status_data
            break
        if status in ("Failed", "Cancelled"):
            poll_err = f"Scan {status}"
            break
        time.sleep(POLL_INTERVAL_SECONDS)
    
    if poll_err or not poll_result:
        print(f"  Batch {batch_idx+1}: Poll failed - {poll_err or 'timeout'}")
        continue
    
    # Get scan result
    result_url = f"{PBI_BASE}/v1.0/myorg/admin/workspaces/scanResult/{scan_id}"
    scan_result, result_err = api_request("GET", result_url)
    if result_err:
        print(f"  Batch {batch_idx+1}: Result failed - {result_err}")
        continue
    
    # Search for target datasource IDs
    for ws in scan_result.get("workspaces", []):
        ws_id = ws.get("id")
        ws_name = ws.get("name", "")
        
        # Check if any top-level datasourceInstances match
        ws_ds_instances = ws.get("datasourceInstances") or scan_result.get("datasourceInstances", [])
        matched_ds = [
            ds for ds in ws_ds_instances
            if (ds.get("datasourceId") or "").lower() in target_set
        ]
        
        # Check artifact datasourceUsages
        artifact_collections = [
            ("datasets", "Dataset"), ("dataflows", "Dataflow"),
            ("reports", "Report"), ("dashboards", "Dashboard"),
            ("lakehouses", "Lakehouse"), ("warehouses", "Warehouse"),
            ("datamarts", "Datamart"), ("notebooks", "Notebook"),
            ("pipelines", "Pipeline"), ("semanticModels", "SemanticModel"),
        ]
        
        ws_has_match = len(matched_ds) > 0
        for coll_key, art_type in artifact_collections:
            for artifact in ws.get(coll_key, []):
                usages = artifact.get("datasourceUsages", [])
                misconfig = artifact.get("misconfiguredDatasourceUsages", [])
                all_usages = usages + misconfig
                matched_usage_ids = [
                    u.get("datasourceInstanceId")
                    for u in all_usages
                    if (u.get("datasourceInstanceId") or "").lower() in target_set
                ]
                if matched_usage_ids:
                    ws_has_match = True
                    matching_artifacts.append({
                        "workspace_id": ws_id,
                        "workspace_name": ws_name,
                        "artifact_id": artifact.get("id") or artifact.get("objectId"),
                        "artifact_name": artifact.get("name") or artifact.get("displayName"),
                        "artifact_type": art_type,
                        "datasource_instance_ids": matched_usage_ids,
                        "is_misconfigured": any(
                            (u.get("datasourceInstanceId") or "").lower() in target_set
                            for u in misconfig
                        ),
                    })
        
        if ws_has_match:
            ws_with_matches += 1
            matching_workspaces_raw.append({
                "workspace_id": ws_id,
                "workspace_name": ws_name,
                "workspace_raw_json": json.dumps(ws),
                "matched_datasource_instances": json.dumps(matched_ds),
                "scanned_at": utc_now().isoformat(),
            })
            workspace_datasource_map[ws_id] = matched_ds
    
    ws_scanned += len(batch_ids)
    
    if (batch_idx + 1) % 3 == 0 or batch_idx == total_batches - 1:
        print(f"  [scan] Batch {batch_idx+1}/{total_batches} | "
              f"Scanned: {ws_scanned}/{len(workspace_ids)} | "
              f"Workspaces with matches: {ws_with_matches} | "
              f"Artifacts matched: {len(matching_artifacts)}")
    
    if batch_idx < total_batches - 1:
        time.sleep(SCAN_PAUSE_SECONDS)

print(f"\n{'='*60}")
print(f"  SCANNER PHASE COMPLETE")
print(f"{'='*60}")
print(f"  Workspaces scanned: {ws_scanned}")
print(f"  Workspaces with matching datasources: {ws_with_matches}")
print(f"  Artifacts using target datasource IDs: {len(matching_artifacts)}")
print(f"{'='*60}")


StatementMeta(, 03250aa0-f0f9-495f-9be1-46982cff225c, 7, Finished, Available, Finished, False)

Scanning 73 workspaces in 1 batches...
Looking for datasource IDs: ['92b5963e-05c1-4d87-94cb-96625efec6e5', 'b1c0fcef-d98e-4cc9-9025-d76bda9ddbc2', 'acf72318-30ed-4e8b-965e-392babdabdcb']


  [scan] Batch 1/1 | Scanned: 73/73 | Workspaces with matches: 0 | Artifacts matched: 0

  SCANNER PHASE COMPLETE
  Workspaces scanned: 73
  Workspaces with matching datasources: 0
  Artifacts using target datasource IDs: 0


In [ ]:
# ============================================================
# Step 3: getDefinition for matching artifacts to find source/sink details
# ============================================================

GUID_RE = re.compile(r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}', re.IGNORECASE)

SINK_KEYWORDS = {"sink", "destination", "output", "write", "copy into", "target"}
SOURCE_KEYWORDS = {"source", "input", "read", "origin", "from"}

EXTERNAL_PATTERNS = [
    r"\.database\.windows\.net",
    r"\.blob\.core\.windows\.net",
    r"\.dfs\.core\.windows\.net",
    r"\.documents\.azure\.com",
    r"s3\.amazonaws\.com",
    r"storage\.googleapis\.com",
]

def infer_direction(context_text):
    """Infer if a reference is source or sink based on surrounding context."""
    text = (context_text or "").lower()
    is_sink = any(kw in text for kw in SINK_KEYWORDS)
    is_source = any(kw in text for kw in SOURCE_KEYWORDS)
    if is_sink and not is_source:
        return "sink"
    if is_source and not is_sink:
        return "source"
    if is_sink and is_source:
        return "both"
    return "unknown"

def extract_connection_details(definition_text, target_ds_ids):
    """Extract connection details and direction from artifact definition."""
    findings = []
    if not definition_text:
        return findings
    
    try:
        parsed = json.loads(definition_text)
    except (json.JSONDecodeError, ValueError):
        parsed = None
    
    # Strategy 1: Parse structured JSON (pipelines, dataflows)
    if parsed:
        findings.extend(_scan_json_recursive(parsed, target_ds_ids, path="$"))
    
    # Strategy 2: Regex scan for URLs, server names, connection strings
    url_re = re.compile(r'(?:https?|ftp|sftp|abfss?|wasbs?|jdbc)://[^\s"<>)\]]+', re.IGNORECASE)
    conn_str_re = re.compile(r'(?:Server|Data Source|AccountName|DefaultEndpointsProtocol)=[^;\n"]{5,}', re.IGNORECASE)
    
    for match in url_re.finditer(definition_text):
        url = match.group(0)
        context = definition_text[max(0, match.start()-200):min(len(definition_text), match.end()+200)]
        direction = infer_direction(context)
        findings.append({
            "detail_type": "url",
            "value": url[:500],
            "direction": direction,
            "context_snippet": context[:400],
        })
    
    for match in conn_str_re.finditer(definition_text):
        cs = match.group(0)
        context = definition_text[max(0, match.start()-200):min(len(definition_text), match.end()+200)]
        direction = infer_direction(context)
        findings.append({
            "detail_type": "connection_string",
            "value": cs[:500],
            "direction": direction,
            "context_snippet": context[:400],
        })
    
    return findings

def _scan_json_recursive(obj, target_ds_ids, path="$", depth=0):
    """Recursively scan JSON for connection references and extract details."""
    findings = []
    if depth > 20:
        return findings
    
    if isinstance(obj, dict):
        # Check for activity-level source/sink patterns (pipeline JSON)
        obj_type = (obj.get("type") or "").lower()
        obj_str = json.dumps(obj)
        
        # Check if this node references any target datasource ID
        has_target_ref = any(ds_id.lower() in obj_str.lower() for ds_id in target_ds_ids)
        
        if has_target_ref or any(k in obj for k in ["connectionReference", "connectionReferences", "linkedServiceName"]):
            # Extract connection info
            conn_ref = obj.get("connectionReference") or {}
            if isinstance(conn_ref, dict):
                conn_type = conn_ref.get("type") or conn_ref.get("connectionType") or ""
                conn_id = conn_ref.get("connectionId") or conn_ref.get("id") or ""
                
                # Determine direction from parent context
                direction = infer_direction(path + " " + obj_type)
                
                # Look for server/database/url in typeProperties
                type_props = obj.get("typeProperties") or {}
                server = type_props.get("server") or type_props.get("host") or ""
                database = type_props.get("database") or type_props.get("catalog") or ""
                url = type_props.get("url") or type_props.get("endpoint") or ""
                
                findings.append({
                    "detail_type": "connection_reference",
                    "value": json.dumps({
                        "connection_type": conn_type,
                        "connection_id": conn_id,
                        "server": server,
                        "database": database,
                        "url": url,
                    })[:500],
                    "direction": direction,
                    "context_snippet": json.dumps(obj)[:400],
                })
        
        # Check source/sink blocks in pipeline activities
        for key in ["source", "sink"]:
            if key in obj and isinstance(obj[key], dict):
                source_sink = obj[key]
                ss_type = source_sink.get("type", "")
                ss_props = source_sink.get("typeProperties") or source_sink
                server = ss_props.get("server") or ss_props.get("host") or ""
                database = ss_props.get("database") or ""
                url_val = ss_props.get("url") or ss_props.get("endpoint") or ""
                conn_ref = source_sink.get("connectionReference") or {}
                
                findings.append({
                    "detail_type": f"pipeline_{key}",
                    "value": json.dumps({
                        "type": ss_type,
                        "server": server,
                        "database": database,
                        "url": url_val,
                        "connection_id": conn_ref.get("connectionId") or conn_ref.get("id") or "",
                        "connection_type": conn_ref.get("type") or "",
                    })[:500],
                    "direction": key,
                    "context_snippet": json.dumps(source_sink)[:400],
                })
        
        # Recurse into children
        for k, v in obj.items():
            if isinstance(v, (dict, list)):
                findings.extend(_scan_json_recursive(v, target_ds_ids, f"{path}.{k}", depth+1))
    
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            if isinstance(item, (dict, list)):
                findings.extend(_scan_json_recursive(item, target_ds_ids, f"{path}[{i}]", depth+1))
    
    return findings


# Filter to scannable items only
scannable_artifacts = [
    a for a in matching_artifacts
    if a["artifact_type"] in SCANNABLE_ITEM_TYPES
]
non_scannable = [a for a in matching_artifacts if a["artifact_type"] not in SCANNABLE_ITEM_TYPES]

print(f"Artifacts to scan via getDefinition: {len(scannable_artifacts)}")
print(f"Artifacts without definition scanning (datasets/reports/etc): {len(non_scannable)}")

# Scan definitions in batches
item_details = []
scan_success = 0
scan_failed = 0
scan_errors = []

total_def_batches = math.ceil(len(scannable_artifacts) / DEFINITION_BATCH_SIZE)

for batch_idx in range(total_def_batches):
    batch_start = batch_idx * DEFINITION_BATCH_SIZE
    batch_end = min(batch_start + DEFINITION_BATCH_SIZE, len(scannable_artifacts))
    batch = scannable_artifacts[batch_start:batch_end]
    
    for artifact in batch:
        ws_id = artifact["workspace_id"]
        item_id = artifact["artifact_id"]
        item_name = artifact["artifact_name"]
        item_type = artifact["artifact_type"]
        
        # Trigger getDefinition
        def_url = f"{FABRIC_BASE}/v1/workspaces/{ws_id}/items/{item_id}/getDefinition"
        def_result, def_err = api_request("POST", def_url, body={})
        
        if def_err:
            scan_failed += 1
            scan_errors.append(f"{item_name}: {def_err[:100]}")
            item_details.append({
                "workspace_id": ws_id,
                "workspace_name": artifact["workspace_name"],
                "artifact_id": item_id,
                "artifact_name": item_name,
                "artifact_type": item_type,
                "datasource_instance_ids": json.dumps(artifact["datasource_instance_ids"]),
                "detail_type": "error",
                "value": def_err[:500],
                "direction": "unknown",
                "context_snippet": None,
                "scanned_at": utc_now().isoformat(),
            })
            continue
        
        # Handle LRO (202)
        definition = None
        if hasattr(def_result, 'headers'):
            # It's a Response object (202)
            location = def_result.headers.get("Location")
            if location:
                lro_result, lro_err = poll_lro(location)
                if lro_err:
                    scan_failed += 1
                    scan_errors.append(f"{item_name}: LRO {lro_err[:100]}")
                    continue
                # Get result
                result_url = lro_result.get("resultUrl") or lro_result.get("resultLocation") or f"{location}/result"
                definition, _ = api_request("GET", result_url)
        elif isinstance(def_result, dict):
            definition = def_result
        
        if not definition:
            scan_failed += 1
            continue
        
        # Decode definition parts
        parts = (definition.get("definition") or {}).get("parts") or []
        for part in parts:
            payload_raw = part.get("payload") or part.get("content") or ""
            try:
                payload_text = base64.b64decode(payload_raw).decode("utf-8", errors="replace")
            except Exception:
                payload_text = payload_raw if isinstance(payload_raw, str) else ""
            
            if not payload_text:
                continue
            
            findings = extract_connection_details(payload_text, artifact["datasource_instance_ids"])
            for finding in findings:
                item_details.append({
                    "workspace_id": ws_id,
                    "workspace_name": artifact["workspace_name"],
                    "artifact_id": item_id,
                    "artifact_name": item_name,
                    "artifact_type": item_type,
                    "datasource_instance_ids": json.dumps(artifact["datasource_instance_ids"]),
                    "detail_type": finding["detail_type"],
                    "value": finding["value"],
                    "direction": finding["direction"],
                    "context_snippet": finding.get("context_snippet"),
                    "scanned_at": utc_now().isoformat(),
                })
        
        scan_success += 1
    
    if (batch_idx + 1) % 3 == 0 or batch_idx == total_def_batches - 1:
        print(f"  [definitions] Batch {batch_idx+1}/{total_def_batches} | "
              f"Success: {scan_success} | Failed: {scan_failed} | "
              f"Details found: {len(item_details)}")
    
    if batch_idx < total_def_batches - 1:
        time.sleep(DEFINITION_PAUSE_SECONDS)

# Add non-scannable artifacts (datasets/reports) with datasource info from Scanner API
for artifact in non_scannable:
    ws_id = artifact["workspace_id"]
    ds_details = workspace_datasource_map.get(ws_id, [])
    matched_ds = [
        ds for ds in ds_details
        if (ds.get("datasourceId") or "").lower() in set(d.lower() for d in artifact["datasource_instance_ids"])
    ]
    for ds in matched_ds:
        conn_details = ds.get("connectionDetails", {})
        item_details.append({
            "workspace_id": ws_id,
            "workspace_name": artifact["workspace_name"],
            "artifact_id": artifact["artifact_id"],
            "artifact_name": artifact["artifact_name"],
            "artifact_type": artifact["artifact_type"],
            "datasource_instance_ids": json.dumps(artifact["datasource_instance_ids"]),
            "detail_type": "scanner_datasource_instance",
            "value": json.dumps({
                "datasource_type": ds.get("datasourceType"),
                "gateway_id": ds.get("gatewayId"),
                "server": conn_details.get("server"),
                "database": conn_details.get("database"),
                "url": conn_details.get("url"),
                "path": conn_details.get("path"),
                "account": conn_details.get("account"),
                "domain": conn_details.get("domain"),
                "extensionDataSourceKind": conn_details.get("extensionDataSourceKind"),
                "extensionDataSourcePath": conn_details.get("extensionDataSourcePath"),
            })[:500],
            "direction": "source",  # Scanner API artifacts are always sources
            "context_snippet": json.dumps(conn_details)[:400],
            "scanned_at": utc_now().isoformat(),
        })
    if not matched_ds:
        item_details.append({
            "workspace_id": ws_id,
            "workspace_name": artifact["workspace_name"],
            "artifact_id": artifact["artifact_id"],
            "artifact_name": artifact["artifact_name"],
            "artifact_type": artifact["artifact_type"],
            "datasource_instance_ids": json.dumps(artifact["datasource_instance_ids"]),
            "detail_type": "scanner_usage_reference",
            "value": json.dumps({"note": "Referenced via datasourceUsages but no instance details available"}),
            "direction": "source",
            "context_snippet": None,
            "scanned_at": utc_now().isoformat(),
        })

print(f"\n{'='*60}")
print(f"  DEFINITION SCAN COMPLETE")
print(f"{'='*60}")
print(f"  Items scanned (getDefinition): {scan_success}")
print(f"  Items failed: {scan_failed}")
print(f"  Non-scannable artifacts (from Scanner API): {len(non_scannable)}")
print(f"  Total detail records: {len(item_details)}")
if scan_errors:
    print(f"  Errors (first 5):")
    for e in scan_errors[:5]:
        print(f"    {e}")
print(f"{'='*60}")


StatementMeta(, 03250aa0-f0f9-495f-9be1-46982cff225c, 8, Finished, Available, Finished, False)

Artifacts to scan via getDefinition: 0
Artifacts without definition scanning (datasets/reports/etc): 0

  DEFINITION SCAN COMPLETE
  Items scanned (getDefinition): 0
  Items failed: 0
  Non-scannable artifacts (from Scanner API): 0
  Total detail records: 0


In [ ]:
# ============================================================
# Step 4: Write results to Delta tables
# ============================================================

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# Table 1: Raw workspace data (only matching workspaces)
if matching_workspaces_raw:
    ws_schema = T.StructType([
        T.StructField("workspace_id", T.StringType()),
        T.StructField("workspace_name", T.StringType()),
        T.StructField("workspace_raw_json", T.StringType()),
        T.StructField("matched_datasource_instances", T.StringType()),
        T.StructField("scanned_at", T.StringType()),
    ])
    ws_df = spark.createDataFrame(matching_workspaces_raw, schema=ws_schema)
    ws_df = ws_df.withColumn("scanned_at", F.to_timestamp("scanned_at"))
    ws_df.write.mode("overwrite").format("delta").saveAsTable(RAW_WORKSPACE_TABLE)
    print(f"Wrote {ws_df.count()} rows to {RAW_WORKSPACE_TABLE}")
else:
    print(f"No matching workspaces found - {RAW_WORKSPACE_TABLE} not written")

# Table 2: Item-level datasource details
if item_details:
    item_schema = T.StructType([
        T.StructField("workspace_id", T.StringType()),
        T.StructField("workspace_name", T.StringType()),
        T.StructField("artifact_id", T.StringType()),
        T.StructField("artifact_name", T.StringType()),
        T.StructField("artifact_type", T.StringType()),
        T.StructField("datasource_instance_ids", T.StringType()),
        T.StructField("detail_type", T.StringType()),
        T.StructField("value", T.StringType()),
        T.StructField("direction", T.StringType()),
        T.StructField("context_snippet", T.StringType()),
        T.StructField("scanned_at", T.StringType()),
    ])
    item_df = spark.createDataFrame(item_details, schema=item_schema)
    item_df = item_df.withColumn("scanned_at", F.to_timestamp("scanned_at"))
    item_df.write.mode("overwrite").format("delta").saveAsTable(ITEM_DETAILS_TABLE)
    print(f"Wrote {item_df.count()} rows to {ITEM_DETAILS_TABLE}")
else:
    print(f"No item details found - {ITEM_DETAILS_TABLE} not written")

print(f"\nDone! Tables: {RAW_WORKSPACE_TABLE}, {ITEM_DETAILS_TABLE}")


StatementMeta(, 03250aa0-f0f9-495f-9be1-46982cff225c, 9, Finished, Available, Finished, False)

No matching workspaces found - scanner_workspace_raw not written
No item details found - scanner_item_datasource_details not written

Done! Tables: scanner_workspace_raw, scanner_item_datasource_details


In [ ]:
# ============================================================
# Step 5: Summary & display
# ============================================================

print(f"{'='*60}")
print(f"  FINAL REPORT")
print(f"{'='*60}")
print(f"  Target datasource IDs: {TARGET_DATASOURCE_IDS}")
print(f"  Workspaces scanned: {ws_scanned}")
print(f"  Workspaces with matches: {ws_with_matches}")
print(f"  Artifacts referencing target datasources: {len(matching_artifacts)}")
print(f"  Detail records captured: {len(item_details)}")
print(f"{'='*60}")

if item_details:
    print("\n--- Item Details (from Delta table) ---")
    display(spark.table(ITEM_DETAILS_TABLE))

if matching_workspaces_raw:
    print("\n--- Matching Workspaces ---")
    display(
        spark.table(RAW_WORKSPACE_TABLE)
        .select("workspace_id", "workspace_name", "matched_datasource_instances", "scanned_at")
    )


StatementMeta(, 03250aa0-f0f9-495f-9be1-46982cff225c, 10, Finished, Available, Finished, False)

  FINAL REPORT
  Target datasource IDs: ['92b5963e-05c1-4d87-94cb-96625efec6e5', 'b1c0fcef-d98e-4cc9-9025-d76bda9ddbc2', 'acf72318-30ed-4e8b-965e-392babdabdcb']
  Workspaces scanned: 73
  Workspaces with matches: 0
  Artifacts referencing target datasources: 0
  Detail records captured: 0
